## Setup

Importings libs and setting up data frames and some helper functions...

In [1]:
# Import necessary libraries
import os
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import polars as pl
from enum import IntFlag
from collections import defaultdict
import pandas as pd
from ipywidgets import Dropdown, HBox, interactive_output
import duckdb


plt.style.use('ggplot')
%matplotlib inline

In [2]:
path = r"C:\Users\Kuinox\Documents\parquet_output"
addresses_file = os.path.join(path, 'addresses.parquet')
symbols_file = os.path.join(path, 'symbols.parquet')
tracesamples_file = os.path.join(path, 'tracesamples.parquet')
comms_file = os.path.join(path, 'comms.parquet')
aux_events_file = os.path.join(path, 'aux_events.parquet')

In [3]:
addresses_df = pl.scan_parquet(addresses_file).lazy().set_sorted('id')
symbols_df = pl.scan_parquet(symbols_file).lazy()
tracesamples_df = pl.scan_parquet(tracesamples_file).lazy().set_sorted('time').set_sorted('id')
comms_df = pl.scan_parquet(comms_file).lazy()
aux_events_df = pl.scan_parquet(aux_events_file).lazy()


## Helpers methods

In [ ]:
def df_to_dict(df):
    collected = df.collect()
    return dict(zip(collected['id'], collected['symbol']))



def map_column_values(df, old_col_name, new_col_name, mapping_dict):
    return df.with_columns(
        pl.col(old_col_name).map_elements(
            lambda x: mapping_dict.get(x, "Unknown"), 
            return_dtype=pl.Utf8
        ).alias(new_col_name)
    )

comms_id2str = df_to_dict(comms_df)
sym_id2str = df_to_dict(symbols_df)
addresses_df = map_column_values(addresses_df, 'commStrId', 'command_name', comms_id2str)
addresses_df = map_column_values(addresses_df, 'symStrId', 'symbol_name', sym_id2str)

ColumnNotFoundError: "string" not found

## How the raw data looks like...

In [ ]:
print("addresses_df schema:")
print(addresses_df.collect_schema())
print("\nsymbols_df schema:")
print(symbols_df.collect_schema())
print("\ntracesamples_df schema:")
print(tracesamples_df.collect_schema())
print("\naux_events_df schema:")
print(aux_events_df.collect_schema())


addresses_count = addresses_df.select(pl.len()).collect().item()
symbols_count = symbols_df.select(pl.len()).collect().item()
tracesamples_count = tracesamples_df.select(pl.len()).collect().item()
comms_count = comms_df.select(pl.len()).collect().item()
aux_events_count = aux_events_df.select(pl.len()).collect().item()

threads_summary_df = (tracesamples_df
            .join(addresses_df, left_on='id', right_on='traceId', how='inner')
            .select(['tid', 'commStrId'])
            .filter(pl.col('commStrId') != 0)
            .unique()
            .group_by('tid')
            .agg(
                pl.col('commStrId').map_elements(
                    lambda id_val: comms_id2str.get(id_val.item() if isinstance(id_val, pl.Series) and id_val.len() == 1 else id_val, "Unnamed"),
                    return_dtype=pl.Utf8
                ).implode().alias('command_names_list')
            )
            .with_columns(
                pl.col('command_names_list').list.join("/").alias('command_names_str')
            )
            .select(['tid', 'command_names_str'])
            .collect()
)

tid_to_command_str_map = dict(zip(threads_summary_df['tid'], threads_summary_df['command_names_str']))


print("\nTracesamples data (first 5 rows):")
display(tracesamples_df.head(50).collect())

print("\nAddresses data (first 5 rows):")
display(addresses_df.head(5).collect())

print("\nSymbols data (first 5 rows):")
display(symbols_df.head(5).collect())

print("\nComms data (first 5 rows):")
display(comms_df.head(5).collect())

print("\nAux Events data (first 5 rows):")
display(aux_events_df.head(5).collect())

print(f"\nAddresses count: {addresses_count}")
print(f"Symbols count: {symbols_count}")
print(f"Tracesamples count: {tracesamples_count}")
print(f"Comms count: {comms_count}")
print(f"Aux Events count: {aux_events_count}")

print("\nThreads Summary:")
display(threads_summary_df)
print("\nTID to Command String Map:")
print(tid_to_command_str_map)


addresses_df schema:
Schema([('id', UInt64), ('traceId', UInt64), ('address', UInt64), ('pid', UInt32), ('isIp', Boolean), ('size', UInt32), ('symoff', UInt32), ('symStrId', UInt64), ('symStart', UInt64), ('symEnd', UInt64), ('dso', UInt64), ('symBinding', UInt8), ('is64Bit', UInt8), ('isKernelIp', UInt8), ('buildId', Binary), ('filtered', UInt8), ('commStrId', UInt64), ('priv', UInt64), ('command_name', String), ('symbol_name', String)])

symbols_df schema:
Schema([('id', UInt64), ('string', String)])

tracesamples_df schema:
Schema([('id', UInt64), ('perfId', UInt64), ('pid', UInt32), ('tid', UInt32), ('time', UInt64), ('flags', UInt32), ('cpu', UInt32), ('ip', UInt64), ('addr', UInt64), ('period', UInt64), ('insnCnt', UInt64), ('cycCnt', UInt64), ('weight', UInt64), ('cpumode', UInt8), ('addrCorrelatesSym', UInt8), ('event', String), ('machinePid', UInt32), ('vcpu', UInt32), ('is_branch', Boolean), ('is_call', Boolean), ('is_return', Boolean), ('is_conditional', Boolean), ('is_sysca

ComputeError: parquet: File out of specification: Invalid thrift: protocol error

## Pre Processing 
The purpose of the tool we used before are just to to quickly collect and pack the data in a format accessible.  
Also, it allow you to work on this even if you don't run linux, bare metal, with an intel cpu.

### Checking assumptions

In [ ]:
# are all instruction ordered by time ?
time_order_check = tracesamples_df.select(
    pl.col('time').diff().lt(0).sum().alias('out_of_order_count')
).collect()

out_of_order_count = time_order_check[0, 'out_of_order_count']

if out_of_order_count != 0:
    raise ValueError(f"Found {out_of_order_count} instances where instructions are not ordered by time. Processing halted.")



### Aux Events

While the other events happen from the DLFilter, this one come from directly parsing the file.  
Why ? `perf` need to run on the target machine and allow us to fetch easily symbols infos from an instruction. But it only pass samples to the dlfilter.  
There is an important informations in the events: AUX data loss infos.  
This tell us when intel_pt buffer got saturated and it had to drop events.

In [ ]:
aux_events_df = aux_events_df.with_columns(
    (pl.col('flags') != 0).alias('is_error')
)

aux_data_loss = aux_events_df.filter(pl.col('is_error'))
aux_data_loss_count = aux_data_loss.select(pl.len()).collect().item()
print(f"Aux data loss count: {aux_data_loss_count}")

Aux data loss count: 13


### Samples

In [ ]:
class SampleFlags(IntFlag):
    BRANCH       = 1 << 0
    CALL         = 1 << 1
    RETURN       = 1 << 2
    CONDITIONAL  = 1 << 3
    SYSCALLRET   = 1 << 4
    ASYNC        = 1 << 5
    INTERRUPT    = 1 << 6
    TX_ABORT     = 1 << 7
    TRACE_BEGIN  = 1 << 8
    TRACE_END    = 1 << 9
    IN_TX        = 1 << 10
    VMENTRY      = 1 << 11
    VMEXIT       = 1 << 12

tracesamples_df = tracesamples_df.with_columns([
        (pl.col('flags') & SampleFlags.BRANCH > 0).alias('is_branch'),
        (pl.col('flags') & SampleFlags.CALL > 0).alias('is_call'),
        (pl.col('flags') & SampleFlags.RETURN > 0).alias('is_return'),
        (pl.col('flags') & SampleFlags.CONDITIONAL > 0).alias('is_conditional'),
        (pl.col('flags') & SampleFlags.SYSCALLRET > 0).alias('is_syscallret'),
        (pl.col('flags') & SampleFlags.ASYNC > 0).alias('is_async'),
        (pl.col('flags') & SampleFlags.INTERRUPT > 0).alias('is_interrupt'),
        (pl.col('flags') & SampleFlags.TX_ABORT > 0).alias('is_tx_abort'),
        (pl.col('flags') & SampleFlags.TRACE_BEGIN > 0).alias('is_trace_begin'),
        (pl.col('flags') & SampleFlags.TRACE_END > 0).alias('is_trace_end'),
        (pl.col('flags') & SampleFlags.IN_TX > 0).alias('is_in_tx'),
        (pl.col('flags') & SampleFlags.VMENTRY > 0).alias('is_vmentry'),
        (pl.col('flags') & SampleFlags.VMEXIT > 0).alias('is_vmexit')
])

tracesamples_df = tracesamples_df.with_columns([
    pl.col("tid").alias("threadId"),
    pl.col("pid").alias("processId"),
    pl.col("insnCnt").alias("instructionCount"),
    pl.col("cycCnt").alias("cycleCount"),
    pl.col("addrCorrelatesSym").alias("addressCorrelatesSymbol")
])

flag_infos = (
    tracesamples_df
    .select([
        pl.sum('is_branch').alias('branch_count'),
        pl.sum('is_call').alias('call_count'),
        pl.sum('is_return').alias('return_count'),
        pl.sum('is_conditional').alias('conditional_count'),
        pl.sum('is_syscallret').alias('syscallret_count'),
        pl.sum('is_async').alias('async_count'),
        pl.sum('is_interrupt').alias('interrupt_count'),
        pl.sum('is_tx_abort').alias('tx_abort_count'),
        pl.sum('is_trace_begin').alias('trace_begin_count'),
        pl.sum('is_trace_end').alias('trace_end_count'),
        pl.sum('is_in_tx').alias('in_tx_count'),
        pl.sum('is_vmentry').alias('vmentry_count'),
        pl.sum('is_vmexit').alias('vmexit_count')
    ])
    .collect()
)

instruction_counts = tracesamples_df.select(pl.len()).collect().item()
print(f"Instruction counts: {instruction_counts}")

print("Branch type counts:")
display(flag_infos)
thread_buckets = {
    tid: tracesamples_df.filter(pl.col("tid") == tid)
    for tid in tid_to_command_str_map
}

aux_drops = aux_events_df.filter(pl.col("is_error")).select(["tid", "time"]).collect()

drops_by_tid = defaultdict(list)
for r in aux_drops.rows(named=True):
    drops_by_tid[r["tid"]].append(r["time"])
for tid in drops_by_tid:
    drops_by_tid[tid].sort()

thread_segments = {}
for tid, df in thread_buckets.items():
    times = drops_by_tid.get(tid, [])
    if not times:
        thread_segments[tid] = [df]
    else:
        segments = [df.filter(pl.col("time") < times[0])]
        segments += [
            df.filter((pl.col("time") >= s) & (pl.col("time") < e))
            for s, e in zip(times, times[1:])
        ]
        segments.append(df.filter(pl.col("time") >= times[-1]))
        thread_segments[tid] = segments



Instruction counts: 212496770
Branch type counts:


branch_count,call_count,return_count,conditional_count,syscallret_count,async_count,interrupt_count,tx_abort_count,trace_begin_count,trace_end_count,in_tx_count,vmentry_count,vmexit_count
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
821738,212470735,136304216,0,0,0,0,0,0,0,0,0,0


### Stack Reconstruction

Now we can reconstruct the stack for each instruction series.

In [ ]:
traces_with_addresses = tracesamples_df.join(
    addresses_df, 
    left_on='id', 
    right_on='traceId', 
    how='left', 
    validate='1:1'
)
traces_with_addresses.head(5).explain(optimized=True)

'LEFT JOIN:\nLEFT PLAN ON: [col("id")]\n   WITH_COLUMNS:\n   [col("time").set_sorted(), col("id").set_sorted(), [([(col("flags")) & (1)]) > (0)].alias("is_branch"), [([(col("flags")) & (2)]) > (0)].alias("is_call"), [([(col("flags")) & (4)]) > (0)].alias("is_return"), [([(col("flags")) & (8)]) > (0)].alias("is_conditional"), [([(col("flags")) & (16)]) > (0)].alias("is_syscallret"), [([(col("flags")) & (32)]) > (0)].alias("is_async"), [([(col("flags")) & (64)]) > (0)].alias("is_interrupt"), [([(col("flags")) & (128)]) > (0)].alias("is_tx_abort"), [([(col("flags")) & (256)]) > (0)].alias("is_trace_begin"), [([(col("flags")) & (512)]) > (0)].alias("is_trace_end"), [([(col("flags")) & (1024)]) > (0)].alias("is_in_tx"), [([(col("flags")) & (2048)]) > (0)].alias("is_vmentry"), [([(col("flags")) & (4096)]) > (0)].alias("is_vmexit"), col("tid").alias("threadId"), col("pid").alias("processId"), col("insnCnt").alias("instructionCount"), col("cycCnt").alias("cycleCount"), col("addrCorrelatesSym")

### 

# Overview

In [ ]:
def show_instruction_counts(segments_dict, name_map):
    counts_by_name = {}
    for tid, segments in segments_dict.items():
        name = name_map.get(tid, str(tid))
        counts = []
        for seg in segments:
            total = seg.select(
                pl.sum("instructionCount").alias("total")
            ).collect()[0, "total"]
            counts.append(int(total))
        counts_by_name[name] = counts

    for name, counts in counts_by_name.items():
        print(f"Thread {name}: Instruction count per segment: {', '.join(map(str, counts))}")

show_instruction_counts(thread_segments, tid_to_command_str_map)


# map thread‐names ↔ tids
name_to_tid = {v: k for k, v in tid_to_command_str_map.items()}
thread_names = list(name_to_tid.keys())

# dropdowns
thread_dd  = Dropdown(options=thread_names, description="Thread:")
segment_dd = Dropdown(description="Segment:")

def _update_segments(*_):
    tid = name_to_tid[thread_dd.value]
    segs = thread_segments[tid]
    segment_dd.options = list(range(len(segs)))
    segment_dd.value   = 0

thread_dd.observe(_update_segments, 'value')
_update_segments()

def _plot(thread_name, segment_idx):
    tid = name_to_tid[thread_name]
    df = thread_segments[tid][segment_idx].collect()
    t0 = df["time"].min()
    df = df.with_columns(
        ((pl.col("time") - t0)/1e9).alias("time_s")
    ).to_pandas()

    fig = go.Figure()
    # full‐series instructions line
    fig.add_trace(go.Scatter(
        x=df["time_s"],
        y=df["instructionCount"],
        mode="lines",
        name="Instructions"
    ))

    # only first 100 branch‐event dots
    marker_df = df.head(100)
    fig.add_trace(go.Scatter(
        x=marker_df["time_s"],
        y=marker_df["is_branch"].astype(int),
        mode="markers",
        name="Branch (first 100)"
    ))

    fig.update_layout(
        title=f"{thread_name} — Segment {segment_idx}",
        xaxis_title="Time since segment start (s)",
        yaxis_title="Count / Flag",
        hovermode="x unified"
    )
    fig.show()

ui  = HBox([thread_dd, segment_dd])
out = interactive_output(_plot, {'thread_name': thread_dd, 'segment_idx': segment_dd})
display(ui, out)

Thread .NET Finalizer: Instruction count per segment: 563774, 876066
Thread .NET DebugPipe: Instruction count per segment: 7458
Thread dotnet: Instruction count per segment: 3107064, 6719852, 812456, 1722042, 896842, 4513302, 3900596, 3821196, 2630342, 1196786, 245701189, 4770619, 102859651
Thread .NET Tiered Com: Instruction count per segment: 14362
Thread .NET SynchManag: Instruction count per segment: 7962
Thread .NET SigHandler: Instruction count per segment: 2940
Thread .NET Debugger: Instruction count per segment: 17940
Thread .NET EventPipe: Instruction count per segment: 10460


Output()